# GPT-2 Medium: Factual Recall Baseline

Logit lens + attention/MLP decomposition on known-good facts.
Establishes the method before running domain-specific prompts.

In [ ]:
import sys
sys.path.insert(0, '../../..')

import matplotlib.pyplot as plt
from src.models import load_model, unload
from src.logit_lens import logit_lens, print_logit_lens
from src.decompose import decompose_contributions, print_decomposition, summarize_contributions
from src.heads import per_head_contributions, top_heads, print_top_heads
from src.viz import plot_logit_lens, plot_decomposition, plot_head_heatmap, save_figures

## GPT-2 Medium
24 layers, d_model=1024, d_mlp=4096, 16 heads.

In [2]:
model, info = load_model("gpt2-medium")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-medium into HookedTransformer
Loaded gpt2-medium on mps
  24 layers | 16 heads | d_model=1024 | d_mlp=4096 | sequential attn+MLP


In [3]:
prompts = [
    ("The capital of France is", " Paris", "paris"),
    ("The capital of Germany is", " Berlin", "berlin"),
]

results = {}
for prompt, target, label in prompts:
    print(f"\n{'='*60}")
    print(f"  {label.upper()}: \"{prompt}\" → \"{target}\"")
    print(f"{'='*60}\n")

    df_lens, cache = logit_lens(model, prompt, target)
    print_logit_lens(df_lens)

    df_decomp = decompose_contributions(model, cache, target)
    print_decomposition(df_decomp)
    print(summarize_contributions(df_decomp))

    df_heads = per_head_contributions(model, cache, target)
    print_top_heads(df_heads)

    results[label] = {"lens": df_lens, "decomp": df_decomp, "heads": df_heads}


  PARIS: "The capital of France is" → " Paris"

Model: gpt2-medium
Prompt: "The capital of France is"
Target: " Paris" (token 6342)
Final prediction: " the"

Layer    Top Prediction       Target Rank    Target Prob 
------------------------------------------------------
0         not                 10100          0.000010    
1         not                 6065           0.000010    
2         not                 3391           0.000019    
3         not                 2356           0.000016    
4         not                 1576           0.000026    
5         not                 1670           0.000022    
6         not                 1661           0.000018    
7         not                 1645           0.000027    
8         often               425            0.000160    
9         now                 193            0.000517    
10        now                 335            0.000196    
11        now                 294            0.000111    
12        often               28

In [ ]:
for label, data in results.items():
    print(f"\n--- {label} ---")
    plt.close(plot_logit_lens(data["lens"]))
    plt.close(plot_decomposition(data["decomp"]))
    plt.close(plot_head_heatmap(data["heads"]))

In [ ]:
model_name = "gpt2-medium"

for label, data in results.items():
    data["lens"].to_csv(f"../../results/{model_name}/{label}-logit-lens.csv", index=False)
    data["decomp"].to_csv(f"../../results/{model_name}/{label}-decomposition.csv", index=False)
    data["heads"].to_csv(f"../../results/{model_name}/{label}-heads.csv", index=False)
    save_figures(model_name, label,
                 logit_lens=data["lens"], decomposition=data["decomp"], heads=data["heads"],
                 out_dir=f"../../results/{model_name}")

In [6]:
unload(model)

Model unloaded, memory cleared
